# Lab W7: Klasifikasi Teks — IndoBERT 2×2 Experiment

## Pre-flight Checklist

> [!IMPORTANT]
> Lab ini adalah lab utama W7. Konsep yang ditandai (§) merujuk ke `07_W7_Text_Transformers_Repo_Adoption.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W7 sudah dibaca, terutama §1 (text→token→embedding pipeline), §1.3 (cara kerja attention: QKV, Transformer block, positional encoding, freeze vs fine-tune).
- Familiar dengan HuggingFace `transformers` dan `datasets` library (dasar).
- GPU **sangat dianjurkan** — BERT training di CPU sangat lambat.

**Yang akan Anda hasilkan di akhir lab:**
- Inspect tokenizer IndoBERT pada 10 sampel IndoNLU SmSA.
- 2×2 experiment: frozen vs fine-tune × [CLS] pool vs mean pool.
- Tabel perbandingan macro-F1 keempat kondisi.
- Synthesis note: dua alasan memilih IndoBERT vs alternatif.

**Resource:**
- **Hardware:** GPU wajib untuk fine-tuning (3-5 menit per kondisi). CPU mungkin butuh >30 menit per kondisi.
- **Storage:** ~500 MB (IndoBERT-lite model + cache).
- **Estimasi waktu kerja:** 2-3 jam termasuk training keempat kondisi.

**Pendamping:** Bab W7 di `07_W7_Text_Transformers_Repo_Adoption.md`.

**Tujuan pedagogis:** memahami dampak strategi freeze vs fine-tune dan metode pooling pada performa klasifikasi teks; membaca macro-F1 sebagai metric untuk data tidak seimbang; menulis synthesis note berbasis bukti eksperimental.

## 0. Setup

In [ ]:
# ── Install dependencies (Colab) ─────────────────────────────────────────────
# !pip install transformers datasets accelerate scikit-learn

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

## 1. Muat Dataset IndoNLU SmSA

SmSA (*Sentiment Analysis*) adalah dataset sentimen Bahasa Indonesia — klasifikasi 3 kelas: positif, negatif, netral.

In [ ]:
dataset = load_dataset('indonlp/indonlu', 'smsa')
print('Splits:', list(dataset.keys()))
print('Train size:', len(dataset['train']))
print('Validation size:', len(dataset['validation']))
print('Test size:', len(dataset['test']))

# Label distribution
labels = ['negatif', 'positif', 'netral']  # 0, 1, 2
for split in ['train', 'validation', 'test']:
    counts = np.bincount(dataset[split]['label'], minlength=3)
    pcts = [f'{c/len(dataset[split])*100:.1f}%' for c in counts]
    print(f'  {split:>12s}: {dict(zip(labels, pcts))}')

## 2. Inspeksi Tokenizer IndoBERT

Lihat bagaimana IndoBERT-lite mentokenisasi 10 sampel pertama — termasuk token spesial [CLS], [SEP], dan subword splitting.

In [ ]:
model_name = 'indobenchmark/indobert-lite-base-p1'
tokenizer = AutoTokenizer.from_pretrained(model_name)

print('Tokenizer:', tokenizer.__class__.__name__)
print('Vocab size:', tokenizer.vocab_size)
print('Special tokens:', tokenizer.all_special_tokens)
print()

# Inspeksi 10 sampel
for i in range(10):
    text = dataset['train'][i]['text']
    label = labels[dataset['train'][i]['label']]
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    print(f'--- Sample {i+1} (label: {label}) ---')
    print(f'  Text:   {text[:100]}{"..." if len(text) > 100 else ""}')
    print(f'  Tokens: {tokens}')
    print(f'  IDs:    {ids}')
    print(f'  Decoded: {decoded[:100]}{"..." if len(decoded) > 100 else ""}')
    print(f'  Len:    {len(ids)} tokens')
    print()

In [ ]:
# Distribusi panjang token setelah tokenisasi
all_lengths = []
for example in dataset['train']:
    all_lengths.append(len(tokenizer.encode(example['text'])))

all_lengths = np.array(all_lengths)
print(f'Panjang token — min: {all_lengths.min()}, max: {all_lengths.max()}, mean: {all_lengths.mean():.0f}, median: {np.median(all_lengths):.0f}')

import matplotlib.pyplot as plt
plt.hist(all_lengths, bins=30, edgecolor='black')
plt.xlabel('Jumlah token'); plt.ylabel('Frekuensi')
plt.title('Distribusi panjang token setelah tokenisasi IndoBERT')
plt.axvline(all_lengths.mean(), color='red', linestyle='--', label=f'mean={all_lengths.mean():.0f}')
plt.legend()
plt.show()

## 3. Model Factory: 2×2 Experiment

Buat 4 konfigurasi model:

|            | [CLS] pooling       | Mean pooling        |
| ---------- | ------------------- | ------------------- |
| **Frozen** | `[CLS]` token → head | mean(tokens) → head |
| **Fine-tune** | `[CLS]` token → head | mean(tokens) → head |

Semua konfigurasi pakai backbone IndoBERT-lite yang sama, hanya berbeda di dua axis: parameter frozen vs trainable, dan metode pooling.

In [ ]:
class BERTClassifier(nn.Module):
    """IndoBERT classifier dengan pooling yang bisa dipilih."""
    def __init__(self, model_name, num_classes=3, pool='cls', freeze_backbone=True):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.pool = pool
        hidden = self.bert.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, num_classes),
        )
        if freeze_backbone:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        if self.pool == 'cls':
            pooled = out.last_hidden_state[:, 0, :]     # [CLS] token — indeks 0
        else:  # 'mean'
            # Mean over non-pad tokens
            mask_expanded = attention_mask.unsqueeze(-1).float()
            pooled = (out.last_hidden_state * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        return self.head(pooled)

print('BERTClassifier — configurable pool=[cls|mean], freeze_backbone=[True|False]')

## 4. Data Pipeline

Fungsi bantu untuk tokenisasi batch dan DataLoader.

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 32

def collate_fn(batch):
    texts = [item['text'] for item in batch]
    labels = [item['label'] for item in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels)

train_loader = DataLoader(dataset['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(dataset['validation'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(dataset['test'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 5. Training & Evaluasi

Fungsi generik untuk train 5 epoch + evaluasi macro-F1 pada validation set.

In [ ]:
def train_and_eval(model, train_loader, val_loader, epochs=5, lr=2e-5, label=''):
    model = model.to(device)
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01
    )
    crit = nn.CrossEntropyLoss()
    best_val_f1 = 0.0
    
    for epoch in range(epochs):
        # Train
        model.train()
        total_loss = 0.0
        for input_ids, attention_mask, labels in tqdm(train_loader, desc=f'{label} E{epoch+1}', leave=False):
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
            opt.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = crit(logits, labels)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        
        # Eval
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for input_ids, attention_mask, labels in val_loader:
                input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
                logits = model(input_ids, attention_mask)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())
        
        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        best_val_f1 = max(best_val_f1, macro_f1)
        print(f'{label:>30s} E{epoch+1}/{epochs} | loss={total_loss/len(train_loader):.4f} | val macro-F1={macro_f1:.4f}')
    
    return best_val_f1, model

In [ ]:
print('=== 2×2 Experiment — 4 kondisi ===\n')

results = {}

for freeze in [True, False]:
    for pool in ['cls', 'mean']:
        freeze_label = 'FROZEN' if freeze else 'FINETUNE'
        pool_label = '[CLS]' if pool == 'cls' else 'mean'
        label = f'{freeze_label} + {pool_label}'
        print(f'\n--- {label} ---')
        model = BERTClassifier(model_name, num_classes=3, pool=pool, freeze_backbone=freeze)
        best_f1, _ = train_and_eval(model, train_loader, val_loader, epochs=5, label=label)
        results[(freeze_label, pool_label)] = best_f1
        print(f'  => Best val macro-F1: {best_f1:.4f}')

## 6. Tabel Perbandingan 2×2

Susun hasil dalam format matriks 2×2: baris = strategi training, kolom = metode pooling.

In [ ]:
import pandas as pd

tbl = pd.DataFrame([
    {'Strategi': 'Frozen',  'Pooling': '[CLS]', 'Macro-F1': results[('FROZEN', '[CLS]')]},
    {'Strategi': 'Frozen',  'Pooling': 'mean',  'Macro-F1': results[('FROZEN', 'mean')]},
    {'Strategi': 'Fine-tune', 'Pooling': '[CLS]', 'Macro-F1': results[('FINETUNE', '[CLS]')]},
    {'Strategi': 'Fine-tune', 'Pooling': 'mean',  'Macro-F1': results[('FINETUNE', 'mean')]},
])

# Pivot ke 2×2
pivot = tbl.pivot(index='Strategi', columns='Pooling', values='Macro-F1')
print('=== 2×2 Experiment: Val Macro-F1 ===')
print(pivot.to_string(float_format='{:.4f}'.format))

# Bar chart visual
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
vals = tbl['Macro-F1'].values
bars = ax.bar(x, vals, color=['steelblue','lightsteelblue','tomato','lightsalmon'])
ax.set_xticks(x)
ax.set_xticklabels(tbl['Strategi'] + ' + ' + tbl['Pooling'])
ax.set_ylabel('Val Macro-F1')
ax.set_title('2×2 Experiment: Frozen/Fine-tune × [CLS]/Mean Pool')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.4f}', ha='center', va='bottom')
ax.set_ylim(0, max(vals) * 1.2)
plt.tight_layout()
plt.show()

### Analisis Hasil

Jawab pertanyaan berikut berdasarkan data di tabel:

1. **Strategi mana yang terbaik?** (Frozen atau Fine-tune? [CLS] atau mean pool? Kombinasi mana?)
2. **Apakah fine-tune selalu lebih baik?** Bandingkan frozen-[CLS] vs fine-tune-mean — apakah selalu benar bahwa fine-tune > frozen?
3. **Apakah mean pool selalu lebih baik dari [CLS]?** Bandingkan frozen-mean vs frozen-[CLS] dan fine-tune-mean vs fine-tune-[CLS].
4. **Efek interaksi.** Apakah keunggulan fine-tune lebih besar dengan pool tertentu? Atau sama? (Lihat selisih horizontal vs vertikal).

## 7. Synthesis Note

Tulis synthesis note yang menjawab: **dua alasan memilih IndoBERT-lite dibanding alternatif lain** untuk tugas klasifikasi sentimen Bahasa Indonesia. Gunakan bukti dari eksperimen ini DAN pengetahuan dari bacaan W7.

Template synthesis note:

---
### Synthesis: Mengapa IndoBERT-lite untuk Klasifikasi Sentimen Bahasa Indonesia

**Alasan 1:** *[berbasis bukti dari eksperimen — mis. performa macro-F1, efisiensi fine-tuning, dll.]*

**Alasan 2:** *[berbasis pengetahuan dari W7 — mis. pretraining corpus, arsitektur, ekosistem, dll.]*
---

Tulis di sel di bawah:

### Jawaban Synthesis Note

**Alasan 1:**

> *[tulis di sini]*

**Alasan 2:**

> *[tulis di sini]*


### Jawaban Analisis §6

**1. Strategi terbaik:**

> *[tulis di sini]*

**2. Apakah fine-tune selalu lebih baik?**

> *[tulis di sini]*

**3. Apakah mean pool selalu lebih baik?**

> *[tulis di sini]*

**4. Efek interaksi:**

> *[tulis di sini]*

## 8. Evaluasi Test Set (Kondisi Terbaik)

Pilih kondisi terbaik dari 2×2 experiment dan evaluasi pada test set.

In [ ]:
# Latih ulang model terbaik pada train+val, evaluasi pada test set
best_config = max(results, key=results.get)
print(f'Best config: {best_config[0]} + {best_config[1]} (macro-F1={results[best_config]:.4f})')

# Training pada full train (tanpa val split untuk final evaluation)
best_freeze = best_config[0] == 'FROZEN'
best_pool = 'cls' if best_config[1] == '[CLS]' else 'mean'

final_model = BERTClassifier(model_name, num_classes=3, pool=best_pool, freeze_backbone=best_freeze)
final_model = final_model.to(device)

# Gunakan dataset['train'] untuk training (validation loader dipakai untuk early stopping manual)
opt = torch.optim.AdamW(
    [p for p in final_model.parameters() if p.requires_grad], lr=2e-5, weight_decay=0.01
)
crit = nn.CrossEntropyLoss()

for epoch in range(5):
    final_model.train()
    for input_ids, attention_mask, labels in tqdm(train_loader, desc=f'Final E{epoch+1}', leave=False):
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        opt.zero_grad()
        loss = crit(final_model(input_ids, attention_mask), labels)
        loss.backward()
        opt.step()

# Evaluasi pada test set
final_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        logits = final_model(input_ids, attention_mask)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('\n=== Test Set Evaluation ===')
print(f'Accuracy: {np.mean(np.array(all_preds) == np.array(all_labels)):.4f}')
print(f'Macro-F1: {f1_score(all_labels, all_preds, average="macro"):.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=labels, digits=4))

## Self-Check Quick

- [ ] Tokenizer inspection dengan 10+ sampel selesai.
- [ ] Distribusi panjang token diplot.
- [ ] 4 kondisi 2×2 experiment trained, macro-F1 tercatat.
- [ ] Tabel 2×2 pivot dihasilkan.
- [ ] Analisis hasil: 4 pertanyaan terjawab.
- [ ] Synthesis note: 2 alasan memilih IndoBERT.
- [ ] Test set evaluation dengan model terbaik.
- [ ] **Lab 6b (Transformer-mini) selesai — WAJIB untuk Breadth Check.**